In [1]:
import geopandas as gpd
import pandas as pd
import geoplanar
import numpy as np

source: https://dataforsyningen.dk/data/3829
https://ftp.sdfe.dk/main.html?download&weblink=ca102693c712ad4159e4a6f343da60d5

In [2]:
gdf = gpd.read_file("/data/uscuni-ulce/extension2/dk/building_inspire.gpkg")

/home/eliska/extension2/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:200: RuntimeWarning: GPKG: bad application_id=0x00000000 on '/data/uscuni-ulce/extension2/dk/building_inspire.gpkg'
  return ogr_read(


Filter out point geometries - it is mostly wind turbines

In [3]:
gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])]

In [4]:
gdf

,inspireid,dateofrenovation,dateofdemolition,dateofconstruction,conditionofconstruction,beginlifespanversion,endlifespanversion,numberoffloorsaboveground,numberofbuildingunits,numberofdwellings,...,horizontalgeorefaccuracy_uom,verticalgeorefaccuracy,verticalgeorefaccuracy_uom,elevation,heightaboveground,name,externalreferencereference,externalreferenceinfosystem,externalreferenceinfosystemname,geometry
0,https://geo.data.gov.dk/dataset/3237298e-d65b-...,NaT,NaT,NaT,functional,2015-11-27 13:27:31+00:00,NaT,NaN,NaN,NaN,...,m,0.15,m,None,None,None,18D8F50D-38C3-4295-A123-0A9CD0D62DCC,https://datafordeler.dk,"Datafordeler,Bygnings- og Boligregisteret (BBR)","MULTIPOLYGON (((712752.81 6191310.13, 712748.5..."
1,https://geo.data.gov.dk/dataset/3237298e-d65b-...,NaT,NaT,NaT,functional,2015-11-27 13:27:31+00:00,NaT,NaN,NaN,NaN,...,m,0.15,m,None,None,None,7761E035-6EF9-4E3D-861B-EEFE07879B2A,https://datafordeler.dk,"Datafordeler,Bygnings- og Boligregisteret (BBR)","MULTIPOLYGON (((712749.08 6191321.46, 712747.8..."
2,https://geo.data.gov.dk/dataset/3237298e-d65b-...,NaT,NaT,NaT,functional,2015-11-27 13:27:31+00:00,NaT,NaN,NaN,NaN,...,m,0.15,m,None,None,None,A9C6CC07-45FA-48CD-AC15-0EE59D8D0C91,https://datafordeler.dk,"Datafordeler,Bygnings- og Boligregisteret (BBR)","MULTIPOLYGON (((712742.9 6191332.55, 712741.62..."
3,https://geo.data.gov.dk/dataset/3237298e-d65b-...,NaT,NaT,NaT,functional,2015-11-27 13:27:31+00:00,NaT,NaN,NaN,NaN,...,m,0.15,m,None,None,None,F2CE8614-31DE-4F75-B89C-00F243411809,https://datafordeler.dk,"Datafordeler,Bygnings- og Boligregisteret (BBR)","MULTIPOLYGON (((708963.33 6191959.27, 708964.8..."
4,https://geo.data.gov.dk/dataset/3237298e-d65b-...,NaT,NaT,NaT,functional,2015-11-27 13:27:31+00:00,NaT,NaN,NaN,NaN,...,m,0.15,m,None,None,None,9A94D7C2-7613-497C-9EE3-132147B314B0,https://datafordeler.dk,"Datafordeler,Bygnings- og Boligregisteret (BBR)","MULTIPOLYGON (((708963.06 6191967.1, 708961.1 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5991343,https://geo.data.gov.dk/dataset/3237298e-d65b-...,NaT,NaT,NaT,functional,2026-07-22 23:29:58.108000+00:00,NaT,NaN,NaN,NaN,...,m,0.20,m,None,None,None,acdb4f83-a232-41d9-b4fb-72a67b987322,https://datafordeler.dk,"Datafordeler,Bygnings- og Boligregisteret (BBR)","MULTIPOLYGON (((694684.99 6170087.19, 694690.4..."
5991344,https://geo.data.gov.dk/dataset/3237298e-d65b-...,NaT,NaT,NaT,projected,2026-07-27 12:35:05.227000+00:00,NaT,NaN,NaN,NaN,...,m,10.00,m,None,None,None,31595393-a2b7-4146-8b14-ae1096ce5811,https://datafordeler.dk,"Datafordeler,Bygnings- og Boligregisteret (BBR)","MULTIPOLYGON (((681357.06 6150231.75, 681360.0..."
5991345,https://geo.data.gov.dk/dataset/3237298e-d65b-...,NaT,NaT,NaT,projected,2026-07-27 12:35:05.227000+00:00,NaT,NaN,NaN,NaN,...,m,10.00,m,None,None,None,549abc1a-9f36-44a6-837c-8943ac917443,https://datafordeler.dk,"Datafordeler,Bygnings- og Boligregisteret (BBR)","MULTIPOLYGON (((681361.47 6150221.03, 681362.2..."
5991346,https://geo.data.gov.dk/dataset/3237298e-d65b-...,NaT,NaT,NaT,declined,2026-07-27 12:47:28.635000+00:00,NaT,NaN,NaN,NaN,...,m,0.15,m,None,None,None,NaN,NaN,NaN,"MULTIPOLYGON (((701936.23 6206288.65, 701935.7..."


Filter to actually existing buildings

In [5]:
gdf = gdf[gdf["conditionofconstruction"] == "functional"]

Check for weirdly large polygons

In [6]:
gdf["area"] = gdf.geometry.area
largest_100 = gdf.nlargest(100, "area")

In [ ]:
largest_100.explore()

Correct topology is the key, check for overlaps

In [7]:
pairs = geoplanar.overlaps(gdf)
overlap_idx = np.unique(pairs)

In [8]:
gdf_overlaps = gdf.iloc[overlap_idx]

In [ ]:
gdf_overlaps.explore()

In [9]:
gdf = geoplanar.merge_overlaps(gdf, merge_limit=0, overlap_limit=0)

Age of the building might be useful at some point

In [10]:
gdf_clean = gdf[["geometry", "beginlifespanversion"]].copy()

In [11]:
gdf_clean = gdf_clean.explode(index_parts=False).reset_index(drop=True)

In [12]:
gdf_clean = gdf_clean.to_crs(epsg=3035)

In [13]:
gdf_clean

,beginlifespanversion,geometry
0,2015-11-27 13:27:31+00:00,"POLYGON ((4471264.151 3637631.365, 4471260.048..."
1,2015-11-27 13:27:31+00:00,"POLYGON ((4471260.588 3637642.744, 4471259.344..."
2,2015-11-27 13:27:31+00:00,"POLYGON ((4471254.572 3637653.922, 4471253.327..."
3,2015-11-27 13:27:31+00:00,"POLYGON ((4467483.138 3638339.399, 4467484.61 ..."
4,2015-11-27 13:27:31+00:00,"POLYGON ((4467482.984 3638347.227, 4467481.06 ..."
...,...,...
5942176,2026-07-22 23:29:21.079000+00:00,"POLYGON ((4241313.216 3588775.521, 4241317.223..."
5942177,2026-07-22 23:28:20.118000+00:00,"POLYGON ((4474464.743 3631788.266, 4474465.498..."
5942178,2026-07-22 23:28:03.580000+00:00,"POLYGON ((4292558.829 3532259.226, 4292559.546..."
5942179,2026-07-22 23:27:52.754000+00:00,"POLYGON ((4465272.75 3628696.162, 4465271.807 ..."


In [14]:
gdf_clean.to_parquet("/data/uscuni-ulce/extension2/dk/clean_3035.parquet")